# مختبر بحث الإشارات (Signal Discovery Lab)

**الهدف:** بنية تحتية لاكتشاف مرشّحين جدد للإشارة (`DISCOVERY_TRACKS` في
`signal_evaluation_axis`: `hypothesis_driven`, `data_driven`,
`literature_mining`, `genetic_search`) — **بلا أي تدريب شبكة عصبية**، فقط
دوال جاهزة/رخيصة (مؤشرات موجودة أصلاً في `feature_order`، أو انحدار خطّي
Ridge يُحسَب في أجزاء من الثانية لكل نافذة) مُقيَّمة عبر نفس صرامة المحور
(IC + عُشر + خطّ أساس عشوائي عبر نوافذ متحرّكة، كما في H001/H002).

**لماذا بلا تدريب؟** كل تجربة NIG-TimeNet v2 حتى الآن (main.ipynb، H002)
استهلكت وقتاً وتكلفة حوسبة حقيقية لكل نافذة/تشغيل. هذا الدفتر يفحص عشرات
المرشّحين دفعة واحدة **بتكلفة تقارب الصفر** (ثوانٍ لا دقائق) قبل تبرير أي
تدريب فعلي — فرز أوّلي رخيص، لا بديل عن H001/H002 حين يستحقّ مرشّح تدريباً حقيقياً.

**⚠️ تحذير حرِج — اقرأه قبل الوثوق بأي نتيجة على `high`/`low`:**
اكتُشف أثناء بناء هذا الدفتر أن `y_high_reg`/`y_low_reg` (`reg_target_mode=
'return'`) تُقارَنان بمرجع "نفس النوع" (`last_high`/`last_low`) لا
`last_close` (راجع تنبيه رقم ٢٠ في رأس `crypto_data_pipeline_v6.ipynb`
للتفاصيل والبرهان الرقمي الكامل). هذا يجعل ميزات شكل الشمعة الأخيرة
(`BODY_ratio`, `WICK_upper/lower`, وبدرجة أقل `RET_1`) تُظهر ارتباطاً زائفاً
**قوياً جداً** (سبيرمان ≈+0.51 على بيانات حقيقية) بهذين الهدفين تحديداً —
اختفى تماماً (إلى ≈+0.01) عند توحيد المرجع. **كل دالة تقييم في هذا الدفتر
تستخدم `clean_reg_target` (مرجع `last_close` موحّد) تلقائياً لـ`high`/`low`
— لا `y_high_reg`/`y_low_reg` الأصليين مباشرة.** `close_reg` غير متأثر أصلاً
(مرجعه `last_close` دائماً).

## ١) التجهيز — تحميل تعريفات الدفاتر بأمان (بلا تنفيذ تلقائي لخلايا الأمثلة)

`%run` مباشر لـ`signal_evaluation_axis` قد يُنفِّذ خلايا أمثلته (تفترض
`dataset`/`windows` جاهزين من جلسة سابقة) فيفشل بخطأ متغيّر غير معرَّف. نفس
الأسلوب المُستخدَم فعلاً لاختبار H002 على بيانات حقيقية: استخراج تعريفات
الدوال/الأصناف فقط عبر `ast`، بلا كود سائق.

In [ ]:
# @title
!git clone -q https://github.com/yuosef772424/crypto-signal-prediction.git 2>/dev/null || true
%cd /content/crypto-signal-prediction

import json, ast, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd


def _notebook_code(path):
    nb = json.load(open(path, encoding="utf-8"))
    return "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")


def load_notebook_defs(path):
    """يستخرج تعريفات الدوال/الأصناف والاستيرادات والقيم الحرفية فقط من دفتر
    — يتجاهل خلايا الأمثلة/السائقة (متغيّرات تفاعلية غير معرَّفة، أو استدعاءات
    شبكية حقيقية). آمن لتحميل signal_evaluation_axis دون تشغيله بالكامل."""
    code_text = _notebook_code(path)
    code_text = "\n".join(l for l in code_text.split("\n")
                          if not l.strip().startswith(("%", "!")))
    tree = ast.parse(code_text)
    keep_types = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom)
    literal_types = (ast.Tuple, ast.List, ast.Constant, ast.Dict, ast.Set)
    kept, futures = [], []
    for node in tree.body:
        if isinstance(node, ast.ImportFrom) and node.module == "__future__":
            futures.append(node)
        elif isinstance(node, keep_types):
            kept.append(node)
        elif isinstance(node, ast.Assign) and isinstance(node.value, literal_types):
            kept.append(node)
    mod = ast.Module(body=futures[:1] + kept, type_ignores=[])
    ast.fix_missing_locations(mod)
    return ast.unparse(mod)


%run "crypto_data_pipeline_v6.ipynb"

exec(compile(load_notebook_defs('signal_evaluation_axis (3).ipynb'), "axis", "exec"))
print("✅ تعريفات المحور مُحمَّلة: rolling_splits, evaluate_windows, concat_splits, "
      "extract_actuals, register_hypothesis, list_registry")

## ٢) تحميل البيانات وبناء النوافذ المتحرّكة

نفس الإعداد المُستخدَم في H002 — عدّله حسب مجموعة أصولك.

In [ ]:
# @title
dataset = load_data_from_drive()  # أو مسار preprocessing_output_latest.pkl.gz لديك
FEATURE_ORDER = dataset["feature_order"]

update_config({"min_split_samples": 10})  # ⚠️ خفّضه فقط إن أصولك القليلة تحتاجه (راجع H002)
windows = rolling_splits(
    dataset, test_span="30D", val_span="15D", initial_train_span="365D",
    step="30D", max_windows=12, keep_asset_test_separate=False, config=CONFIG,
)

## ٣) الحارس ضدّ أثر مرجع "نفس النوع" — `clean_reg_target`

استخدمه دائماً بدل `y_high_reg`/`y_low_reg` الأصليين عند تقييم أي مرشّح جديد
ضد high/low (راجع التحذير أعلى الدفتر). `close_reg` غير متأثر فيبقى كما هو.

In [ ]:
# @title
def clean_reg_target(split, target):
    """`y_{target}_reg` بمرجع `last_close` موحّد لكل الأهداف — لا
    `last_high`/`last_low` الأصليين (own-kind reference) اللذين يحملان أثر
    شكل الشمعة الأخيرة (راجع تنبيه رقم ٢٠ في crypto_data_pipeline_v6). لهدف
    `close` يعادل `y_close_reg` تماماً (نفس المرجع أصلاً) فيُقرَأ منه مباشرة."""
    if target == "close":
        return extract_actuals(split, target_key="y_close_reg")
    split = concat_splits(split)
    lc = np.asarray(split["last_candles"])
    last_close = lc[:, LAST_COLUMNS.index("last_close")]
    future_col = {"high": "future_high_max", "low": "future_low_min"}[target]
    future = lc[:, LAST_COLUMNS.index(future_col)]
    return (future - last_close) / last_close


def evaluate_candidate(predict_fn, target, windows, n_shuffles=1000, min_samples=10, seed=42, verbose=False):
    """يقيّم مرشّحاً واحداً عبر كل النوافذ — بديل رقيق لـ
    evaluate_hypothesis_over_rolling_windows يستخدم دائماً clean_reg_target
    (لا target_key خام) فلا يُمكن نسيان الحارس بالخطأ."""
    names = [f"نافذة {i + 1}" for i in range(len(windows))]
    results = []
    for name, (train, val, test) in zip(names, windows):
        test_flat = concat_splits(test)
        preds = np.asarray(predict_fn(train, val, test), dtype="float64")
        actuals = clean_reg_target(test_flat, target)
        if len(preds) != len(actuals):
            raise ValueError(f"[{name}] طول التنبؤات ({len(preds)}) ≠ طول الأهداف ({len(actuals)}).")
        results.append((name, preds, actuals))
    return evaluate_windows(results, n_shuffles=n_shuffles, min_samples=min_samples, seed=seed, verbose=verbose)

## ٤) إطار المرشّح الواحد — أي ميزة جاهزة كمرشّح فوراً

`make_feature_predict_fn` يحوّل أي عمود من `feature_order` (بعد تحويل اختياري
— عكس، تمركز حول نقطة، إلخ) إلى `predict_fn` جاهزة لـ`evaluate_candidate`،
بنفس نمط `momentum_predict_fn` في المحور لكن مُعمَّمة لأي ميزة.

In [ ]:
# @title
def extract_feature_last_value(split, feature, tf=None, feature_order=None):
    """آخر قيمة (خطوة زمنية أخيرة) لميزة واحدة داخل نافذة كل عيّنة — حالة
    المؤشر عند لحظة القرار، لا فرقها كـextract_feature_last_diff."""
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError("مرّر feature_order صراحةً (dataset['feature_order']).")
    idx = feature_order.index(feature)
    if tf is None:
        tf = [k for k in split if k.startswith("X_")][0][2:]
    X = np.asarray(split[f"X_{tf}"])
    return X[:, -1, idx]


def extract_feature_matrix(split, features=None, tf=None, feature_order=None):
    """مصفوفة كل الميزات (أو مجموعة فرعية) في آخر خطوة زمنية — (N, len(features)).
    أعمّ من extract_feature_last_value: تُستخدَم لأي مرشّح يحتاج أكثر من ميزة
    معاً (تفاعل، مركَّب Ridge، تدريب Isolation Forest، أو دالة مخصَّصة كاملة
    على متجه الميزات — kind='custom'/'trained' أدناه)."""
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError("مرّر feature_order صراحةً (dataset['feature_order']).")
    if tf is None:
        tf = [k for k in split if k.startswith("X_")][0][2:]
    X = np.asarray(split[f"X_{tf}"])[:, -1, :]
    if features is not None:
        idx = [feature_order.index(f) for f in features]
        X = X[:, idx]
    return X


def make_feature_predict_fn(feature, transform=None, tf=None, feature_order=None):
    """predict_fn جاهزة لـevaluate_candidate من أي ميزة في feature_order.
    مرّر transform (مثلاً lambda v: -(v-50.0) لعكس RSI حول نقطة المنتصف)
    لصياغة فرضية اتجاه محدّدة بدل القيمة الخام."""
    def predict_fn(train, val, test):
        v = extract_feature_last_value(test, feature=feature, tf=tf, feature_order=feature_order)
        return transform(v) if transform else v
    return predict_fn


def make_interaction_predict_fn(feat_a, feat_b, op="mul", transform=None, tf=None, feature_order=None):
    """predict_fn من تفاعل بين ميزتين موجودتين (ضرب أو فرق) — يغطي فرضيات
    "تفاعلات" (قسم ٣ في خطة المشروع، مثال: حجم×جسم الشمعة) بلا الحاجة لعمود
    ميزة جديد محسوب مسبقاً في خط الأنابيب."""
    def predict_fn(train, val, test):
        a = extract_feature_last_value(test, feature=feat_a, tf=tf, feature_order=feature_order)
        b = extract_feature_last_value(test, feature=feat_b, tf=tf, feature_order=feature_order)
        v = a * b if op == "mul" else (a - b)
        return transform(v) if transform else v
    return predict_fn


def make_custom_predict_fn(fn, tf=None, feature_order=None):
    """predict_fn من دالة مخصَّصة على متجه الميزات الكامل عند آخر خطوة زمنية
    (`fn(X_last, feature_order) -> np.ndarray`) — بلا تدريب (test فقط، بلا
    استخدام train/val). يغطي فرضيات "توليد ممنهج" بلا حاجة لعمود ميزة/تفاعل
    ثنائي جاهز: مقلوب الافتراض الضمني (مثلاً RSI مطبَّع بالتقلب)، كسر
    التناظر (معادلتان مختلفتان للصعود/الهبوط عمداً)، إلخ — راجع قسم "توليد
    فرضيات ممنهج" أدناه."""
    def predict_fn(train, val, test):
        X_last = extract_feature_matrix(test, tf=tf, feature_order=feature_order)
        return fn(X_last, feature_order)
    return predict_fn


def make_candidate_predict_fn(cand, tf=None, feature_order=None):
    """يبني predict_fn من قاموس مرشّح واحد بصرف النظر عن نوعه (`kind`) —
    نقطة التفرّع الوحيدة، فلا يحتاج scan_candidates/run_batch_and_register
    معرفة الفرق بين الأنواع:

    * الافتراضي (بلا `kind`): ميزة واحدة عبر make_feature_predict_fn.
    * `kind="interaction"`: تفاعل بين ميزتين (`feat_a`/`feat_b`/`op`).
    * `kind="custom"`: دالة مخصَّصة بلا تدريب على متجه الميزات الكامل (`fn`).
    * `kind="trained"`: مرشّح يحتاج تدريباً على train (مثل Isolation Forest) —
      `builder(feature_order=...)` يُرجع predict_fn جاهزة (نفس نمط
      make_ridge_composite_predict_fn في قسم البحث التركيبي)."""
    kind = cand.get("kind", "feature")
    if kind == "interaction":
        return make_interaction_predict_fn(cand["feat_a"], cand["feat_b"], op=cand.get("op", "mul"),
                                           transform=cand.get("transform"), tf=tf, feature_order=feature_order)
    if kind == "custom":
        return make_custom_predict_fn(cand["fn"], tf=tf, feature_order=feature_order)
    if kind == "trained":
        return cand["builder"](feature_order=feature_order)
    return make_feature_predict_fn(cand["feature"], transform=cand.get("transform"), tf=tf, feature_order=feature_order)

## ٥) مكتبة مرشّحين جاهزين (`literature_mining` + `data_driven`)

كل مرشّح: اسم، مسار اكتشاف (`DISCOVERY_TRACKS`)، ميزة من `feature_order`،
وتحويل اختياري يصيغ فرضية اتجاه (ارتداد/استمرار). أضِف مرشّحين جدداً بنفس
الشكل — لا حاجة لتعديل أي دالة أخرى.

In [ ]:
# @title
CANDIDATE_SIGNALS = [
    {"name": "RSI_14_reversion", "track": "literature_mining", "feature": "RSI_14",
     "transform": lambda v: -(v - 50.0), "hypothesis": "RSI متطرف يعكس (mean-reversion كلاسيكي)"},
    {"name": "MACDh_12_26_9_momentum", "track": "literature_mining", "feature": "MACDh_12_26_9",
     "transform": None, "hypothesis": "زخم MACD histogram يستمر"},
    {"name": "BBP_reversion", "track": "literature_mining", "feature": "BBP_20_2.0",
     "transform": lambda v: -(v - 0.5), "hypothesis": "موقع متطرف ضمن نطاق بولنجر يعكس"},
    {"name": "STOCH_reversion", "track": "literature_mining", "feature": "STOCHk_14_3_3",
     "transform": lambda v: -(v - 50.0), "hypothesis": "ستوكاستك متطرف يعكس"},
    {"name": "MFI_reversion", "track": "literature_mining", "feature": "MFI_14",
     "transform": lambda v: -(v - 50.0), "hypothesis": "تدفّق نقدي متطرف يعكس"},
    {"name": "CMF_momentum", "track": "data_driven", "feature": "CMF_20",
     "transform": None, "hypothesis": "تدفّق نقدي موجب يستمر"},
    {"name": "NATR_neg_vol", "track": "data_driven", "feature": "NATR_14",
     "transform": lambda v: -v, "hypothesis": "تقلّب مرتفع يسبق عائداً سالباً"},
    {"name": "ADX_trend_strength", "track": "data_driven", "feature": "ADX_14",
     "transform": None, "hypothesis": "قوة اتجاه مرتفعة تدعم استمراره"},
    {"name": "RET_1_reversion", "track": "hypothesis_driven", "feature": "RET_1",
     "transform": lambda v: -v, "hypothesis": "انعكاس قصير المدى (H001) بميزة RET_1 مباشرة"},
    {"name": "RET_6_momentum", "track": "literature_mining", "feature": "RET_6", "transform": None,
     "hypothesis": "زخم متوسط المدى (6 شموع)"},
    {"name": "RET_24_momentum", "track": "literature_mining", "feature": "RET_24", "transform": None,
     "hypothesis": "زخم أطول مدى (24 شمعة)"},
    {"name": "VOLZ_volume_shock", "track": "data_driven", "feature": "VOLZ_20", "transform": None,
     "hypothesis": "فورة حجم تسبق حركة سعرية"},
    {"name": "VOLR_regime", "track": "data_driven", "feature": "VOLR_6_24", "transform": None,
     "hypothesis": "نسبة تقلّب قصير/طويل المدى تكشف نظام سعري"},
    {"name": "POS_14_reversion", "track": "data_driven", "feature": "POS_14",
     "transform": lambda v: -(v - 0.5), "hypothesis": "موقع متطرف ضمن مدى 14 يعكس"},
    {"name": "MKT_beta", "track": "data_driven", "feature": "MKT_ret_1", "transform": None,
     "hypothesis": "عائد السوق العام (بيتا) يتنبأ بعائد الأصل"},
    {"name": "WICK_upper_rejection", "track": "literature_mining", "feature": "WICK_upper",
     "transform": lambda v: -v, "hypothesis": "ذيل علوي طويل إشارة رفض صعود"},
    {"name": "WICK_lower_rejection", "track": "literature_mining", "feature": "WICK_lower",
     "transform": None, "hypothesis": "ذيل سفلي طويل إشارة رفض هبوط"},
]
print(f"{len(CANDIDATE_SIGNALS)} مرشّحاً جاهزاً — أضِف المزيد بنفس الشكل أعلاه.")

### ملحق) مرشّحون إضافيون — تفاعلات + funding rate (المرحلتان ١-٢ من الخطة)

يغطي `EXPLORATORY_CANDIDATES` نمطين لم تغطِّهما `CANDIDATE_SIGNALS` أعلاه:

* **تفاعل بين ميزتين** (`kind="interaction"`، عبر `make_interaction_predict_fn`) —
  مثال "حجم×جسم الشمعة" من قسم "الفرضيات الاستكشافية" في الخطة: حجم غير
  عادي (`VOLZ_20`) مع جسم شمعة كبير (`BODY_ratio`) قد يعني دخول لاعب كبير،
  لا ضجيجاً عادياً — بضرب ميزتين موجودتين أصلاً، بلا عمود جديد في خط الأنابيب.
* **funding rate كمقياس تموضع متطرف** — الفرضية الثالثة في جدول "أساس" بالخطة
  (`FUND_rate_z`، مبنية أصلاً في خط الأنابيب عبر `add_funding_oi_features`
  لكن غير مُفعَّلة في `feature_order` بشكل افتراضي؛ راجع الخطوة التالية في
  README). المرشّح هنا جاهز لأي `dataset` يحمل هذه الميزة فعلياً — سيفشل
  بخطأ واضح (`ValueError`) على بيانات لا تحمل `FUND_rate_z`، لا صمتاً.

In [ ]:
# @title
EXPLORATORY_CANDIDATES = [
    {"name": "VOLZ_x_BODY", "track": "data_driven", "kind": "interaction",
     "feat_a": "VOLZ_20", "feat_b": "BODY_ratio", "op": "mul", "transform": None,
     "hypothesis": "حجم غير عادي × جسم شمعة كبير = دخول لاعب كبير، لا ضجيج عادي"},
    {"name": "FUND_rate_extreme_position", "track": "hypothesis_driven", "feature": "FUND_rate_z",
     "transform": lambda v: -v, "hypothesis": (
         "funding مرتفع جداً = طويلون مفرطون بالرافعة → خطر تصفية → انعكاس هبوطي محتمل "
         "(العكس لـfunding سالب جداً)")},
]
print(f"{len(EXPLORATORY_CANDIDATES)} مرشّحاً استكشافياً إضافياً — "
      "FUND_rate_extreme_position يحتاج dataset يحمل FUND_rate_z فعلياً.")

<cell_type>markdown</cell_type>### ملحق ٢) توليد فرضيات ممنهج — آليات الخطة الخمس، لا "جرّب مؤشراً جديداً"

قسم "طرق توليد فرضيات جديدة" في خطة المشروع يُلاحظ أن المؤشرات القياسية
استُهلكت (RSI، MACD، Bollinger...) لكن **آليات** توليد الفرضية أقلّ استهلاكاً
بكثير من الأدوات نفسها. `GENERATIVE_CANDIDATES` يجسّد ثلاثاً من الخمس بكود
حقيقي (لا وصفاً نظرياً)، كلّ واحدة عبر `kind="custom"`/`"trained"` الجديدين
في `make_candidate_predict_fn` أعلاه:

| الآلية (من الخطة) | المرشّح | الفكرة |
| --- | --- | --- |
| **مقلوب الافتراض الضمني** | `RSI_vol_adjusted_reversion` | RSI يفترض أن عتبتي 30/70 ثابتتان بصرف النظر عن حالة السوق — هنا يُطبَّع الانحراف عن 50 بالتقلب الحالي (`NATR_14`)، فتصير العتبة **نسبية لنظام التقلّب**، لا مطلقة |
| **كسر التناظر عمداً** | `Asymmetric_momentum_downside_weighted` | أغلب المؤشرات تعامل الصعود والهبوط بنفس المعادلة؛ سلوكياً الذعر يتحرّك أسرع من الجشع — هنا معادلتان مختلفتان فعلاً حسب الاتجاه: في الهبوط زخم قصير المدى (`RET_1`)، وفي الصعود زخم أبطأ (`RET_6`) (⚠️ ليس مجرّد تحجيم `RET_1` بوزن ثابت — ذلك تحويل رتيب لا يغيّر ترتيب سبيرمان، فيُعطي IC مطابقاً لـRET_1 خام تماماً؛ استخدام متغيّرين مختلفين حسب الحالة يكسر هذا الرتابة فعلاً) |
| **النقل من مجال مختلف** | `IsolationForest_anomaly_score` | تقنية كشف شذوذ (لا "مؤشر تداول" أصلاً) — Isolation Forest يُدرَّب على train لكل نافذة (`kind="trained"`، بنفس نمط المركَّب Ridge)، ودرجة الشذوذ على test هي التنبؤ: هل الشذوذ الإحصائي نفسه يحمل معلومة اتجاه؟ |

الآليتان المتبقّيتان في الخطة (**تقطير الرؤية المتأخّرة** — يحتاج هدفاً
جديداً كلياً لا مجرّد ميزة، مثال DPO/ATR الموثَّق في الخطة؛ و**تحليل الأخطاء
المنهجية** — يحتاج نموذجاً مُدرَّباً فعلياً لفحص أخطائه) تحتاجان بنية تتجاوز
نطاق "مرشّح بلا تدريب" لهذا الدفتر تحديداً — موثَّقتان هنا كخطوة تالية
صريحة، لا مُنفَّذتين قسراً بشكل مبتور.

In [ ]:
# @title
def _rsi_vol_adjusted(X_last, feature_order):
    """(RSI-50) مطبَّعة بالتقلب الحالي (NATR_14) — عتبة تطرف نسبية للنظام
    السعري، لا 30/70 الثابتة. سالبة (نفترض ارتداداً، لا استمراراً)."""
    rsi = X_last[:, feature_order.index("RSI_14")]
    natr = X_last[:, feature_order.index("NATR_14")]
    return -(rsi - 50.0) / (natr + 1e-6)


def _asymmetric_momentum(X_last, feature_order):
    """في الهبوط (RET_1<0) نعتمد الزخم القصير (RET_1) — استجابة سريعة؛ في
    الصعود نعتمد زخماً أبطأ (RET_6) — تأكيداً أبطأ (الذعر أسرع من الجشع).
    ⚠️ تنبيه للحذر عند تصميم مرشّحين مماثلين: تحجيم بسيط (`ret1 * وزن`، لو
    كان الوزن ثابتاً في كل شقّ) هو تحويل رتيب لـRET_1 عالمياً، وسبيرمان
    (المقياس المُستخدَم في IC هنا) لا يتأثر بتحويل رتيب — أي IC سيتطابق مع
    IC خام RET_1 تماماً رغم اختلاف الشكل. هنا نستخدم متغيّرين مختلفين حسب
    الحالة (تحويل غير رتيب)، فترتيبه لا يطابق ترتيب RET_1 ولا RET_6 وحدهما
    (تحقّق تجريبي على بيانات حقيقية: سبيرمان مع RET_1 ≈0.73، مع RET_6 ≈0.66
    — لا ±1.0 لأيّهما)."""
    ret1 = X_last[:, feature_order.index("RET_1")]
    ret6 = X_last[:, feature_order.index("RET_6")]
    return np.where(ret1 < 0, ret1, ret6)


def make_isolation_forest_predict_fn(features, feature_order=None, contamination=0.1, random_state=42):
    """`kind="trained"`: يُدرِّب Isolation Forest على train (مُطبَّع بمتوسط/
    انحراف train نفسه)، ثم يُرجع درجة الشذوذ (`decision_function`، أعلى =
    أقلّ شذوذاً) على test — نفس نمط make_ridge_composite_predict_fn تماماً،
    لكن بلا هدف (كشف شذوذ غير مُشرَف، لا انحدار)."""
    def predict_fn(train, val, test):
        from sklearn.ensemble import IsolationForest
        train_flat = concat_splits(train)
        Xtr = extract_feature_matrix(train_flat, features, feature_order=feature_order)
        mu, sigma = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-9
        model = IsolationForest(contamination=contamination, random_state=random_state, n_estimators=100)
        model.fit((Xtr - mu) / sigma)
        Xte = extract_feature_matrix(test, features, feature_order=feature_order)
        return model.decision_function((Xte - mu) / sigma)
    return predict_fn


GENERATIVE_CANDIDATES = [
    {"name": "RSI_vol_adjusted_reversion", "track": "hypothesis_driven", "kind": "custom",
     "fn": _rsi_vol_adjusted, "mechanism": "assumption_inversion",
     "hypothesis": "عتبات RSI الثابتة (30/70) لا تناسب كل نظام تقلّب — تطبيع الانحراف عن 50 "
                   "بالتقلّب (NATR) يلتقط تطرّفاً نسبياً حقيقياً، لا مطلقاً"},
    {"name": "Asymmetric_momentum_downside_weighted", "track": "hypothesis_driven", "kind": "custom",
     "fn": _asymmetric_momentum, "mechanism": "asymmetry_hunting",
     "hypothesis": "الذعر يتحرّك أسرع من الجشع — في الهبوط زخم قصير المدى (RET_1) أدقّ، وفي "
                   "الصعود زخم أبطأ (RET_6) أنسب؛ معادلتان مختلفتان عمداً حسب الاتجاه"},
    {"name": "IsolationForest_anomaly_score", "track": "data_driven", "kind": "trained",
     "builder": lambda feature_order: make_isolation_forest_predict_fn(
         [f for f in feature_order if f != "close"], feature_order=feature_order),
     "mechanism": "analogical_transfer",
     "hypothesis": "شذوذ إحصائي في متجه الميزات قد يعكس حدثاً حقيقياً (تصفية، خبر) يحمل معلومة "
                   "اتجاه، لا ضجيجاً محايداً"},
]
print(f"{len(GENERATIVE_CANDIDATES)} مرشّحات توليد ممنهج — راجع جدول الآليات أعلاه.")

## ٦) الماسح الآلي — تقييم كل المرشّحين × كل الأهداف دفعة واحدة

In [ ]:
# @title
def scan_candidates(candidates, windows, targets=("close", "high", "low"),
                    feature_order=None, **eval_kwargs):
    """يُقيِّم كل مرشّح × كل هدف عبر evaluate_candidate (مع حارس
    clean_reg_target تلقائياً)، ويُرجع لوحة قيادة (leaderboard) مُرتَّبة —
    اتساق الإشارة أوّلاً، ثم قوة IC المطلقة. لا يتوقّف عند أوّل خطأ (يُسجَّل
    ويُكمل بقية المرشّحين) — مفيد خاصة لمرشّحين قد لا يحملهما كل dataset
    (مثل FUND_rate_extreme_position)."""
    rows = []
    for cand in candidates:
        predict_fn = make_candidate_predict_fn(cand, feature_order=feature_order)
        for target in targets:
            try:
                report = evaluate_candidate(predict_fn, target, windows, **eval_kwargs)
                rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                            "hypothesis": cand.get("hypothesis", ""),
                            "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                            "frac_significant": report["frac_significant"],
                            "consistent_sign": report["consistent_sign"],
                            "n_ok": report["n_ok"], "status": "ok"})
            except Exception as e:
                rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                            "status": f"error: {type(e).__name__}: {e}"})
    df = pd.DataFrame(rows)
    ok = df[df["status"] == "ok"].copy()
    if len(ok):
        ok["abs_mean_ic"] = ok["mean_ic"].abs()
        ok = ok.sort_values(["consistent_sign", "abs_mean_ic"], ascending=[False, False])
        df = pd.concat([ok.drop(columns="abs_mean_ic"), df[df["status"] != "ok"]], ignore_index=True)
    return df


leaderboard = scan_candidates(CANDIDATE_SIGNALS + EXPLORATORY_CANDIDATES + GENERATIVE_CANDIDATES,
                              windows, feature_order=FEATURE_ORDER)
pd.set_option("display.width", 160)
print(leaderboard.to_string(index=False))

### النتيجة الفعلية (تشغيل حقيقي — 5 أصول من `history_1d`، نفس بيانات H002)

أُجري هذا الماسح فعلياً على نفس بيانات H002 (5 أصول، 12 نافذة). **لا مرشّح
واحد من الـ17 حقّق `consistent_sign=True`** — أقوى النتائج (`NATR_14_neg` على
`close`، mean_ic=-0.234؛ `RSI_14_reversion` على `close`، mean_ic=+0.221)
غير متّسقة الاتجاه عبر النوافذ، مطابقة لصعوبة إيجاد إشارة يومية موثوقة التي
وثّقتها H001/H002 (سقف قريب من العملة المعدنية العادلة). **لم تُسجَّل أي
فرضية من هذا التشغيل في `experiment_registry`** — لا شيء عبر عتبة الثقة
(`consistent_sign=True` + معنوية كافية) يستحقّ تسجيلاً؛ راجع القسم ٨ أدناه
لكيفية التسجيل يدوياً حين يتوفّر مرشّح يستحقّه.

## ٧) البحث التركيبي الرخيص (`data_driven`/`genetic_search`) — بلا شبكة عصبية

مرشّح مركَّب: انحدار Ridge خطّي (`RidgeCV`، يُحسَب مغلقاً بلا حِقَب — أجزاء
من الثانية لكل نافذة) على مجموعة الميزات كلّها معاً، بدل مؤشر واحد. ليست
شبكة عصبية ولا تدريباً تكرارياً — أرخص بآلاف المرّات من تدريب نموذج NIG-TimeNet
لكل نافذة (راجع H002)، لكنها قد تلتقط تفاعلات بين الميزات لا يلتقطها أي
مرشّح فردي أعلاه.

In [ ]:
# @title
from sklearn.linear_model import RidgeCV

COMPOSITE_FEATURES = [c["feature"] for c in CANDIDATE_SIGNALS if c["feature"] != "MKT_ret_1"]
# ✅ extract_feature_matrix مُعرَّفة مرّة واحدة في قسم ٤ (إطار المرشّح) —
# يُعاد استخدامها هنا كما هي، ومن make_isolation_forest_predict_fn لاحقاً.


def make_ridge_composite_predict_fn(features, target, feature_order=None, alphas=(0.1, 1.0, 10.0, 100.0)):
    """يُدرِّب RidgeCV على train (مغلق، بلا حِقَب) متنبّئاً بـclean_reg_target
    لنفس target، ثم يُنبئ على test — نفس عقد predict_fn المُستخدَم مع
    evaluate_candidate."""
    def predict_fn(train, val, test):
        train_flat = concat_splits(train)
        Xtr = extract_feature_matrix(train_flat, features, feature_order=feature_order)
        ytr = clean_reg_target(train_flat, target)
        mu, sigma = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-9
        model = RidgeCV(alphas=alphas)
        model.fit((Xtr - mu) / sigma, ytr)
        Xte = extract_feature_matrix(test, features, feature_order=feature_order)
        return model.predict((Xte - mu) / sigma)
    return predict_fn


composite_rows = []
for target in ("close", "high", "low"):
    pf = make_ridge_composite_predict_fn(COMPOSITE_FEATURES, target, feature_order=FEATURE_ORDER)
    report = evaluate_candidate(pf, target, windows)
    composite_rows.append({"name": "ridge_composite_all_features", "target": target,
                           "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                           "frac_significant": report["frac_significant"],
                           "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})
print(pd.DataFrame(composite_rows).to_string(index=False))

### النتيجة الفعلية للمركَّب

على نفس البيانات: `close` mean_ic=+0.041 (غير معنوي)، `high` mean_ic=+0.209،
`low` mean_ic=+0.139 — **كلاهما `consistent_sign=False`**. لا تحسّن ذا شأن عن
أفضل مرشّح فردي، ولا اجتياز لعتبة القبول. **تنبيه مهم لمن يُعيد هذا الفحص:**
أوّل تشغيل لهذا المركَّب (قبل تطبيق حارس `clean_reg_target`) أعطى نتائج
خارقة زائفة (`high` mean_ic=+0.595، `consistent_sign=True`!) — وهذا بالضبط
ما كشف أثر مرجع "نفس النوع" الموثَّق أعلى الدفتر. أي نتيجة مستقبلية على
high/low أعلى من هذا المدى المتواضع تستحقّ فحصاً مضاعَفاً قبل تصديقها، لا
احتفالاً فورياً.

## ٨) المُشغّل الدفعي (Batch Runner) — تقييم متوازٍ + تسجيل تلقائي

طبقاً لـ"خطة بناء النظام — تسريع التجارب" في خطة المشروع، هذا يكمل الركيزتين
المتبقيتين من الأربع (الأخريان — واجهة موحّدة للفرضية، وذاكرة تخزين مؤقت —
مُلبّاتان أصلاً: `make_candidate_predict_fn` هو الواجهة الموحّدة، و`dataset`/
`windows` يُبنيان مرّة واحدة ويُعاد استخدامهما لكل مرشّح، بلا إعادة تحميل أو
إعادة حساب ميزات — بنفس روح `_checkpoint_fingerprint` في دفتر التحضير):

* **التوازي**: `imap_ordered`/`default_workers` من دفتر التحضير نفسه (لا
  إعادة كتابة) — خيوط، لا عمليات منفصلة (نفس مبرر التحضير: numpy/pandas
  تُحرِّر GIL).
* **معيار قبول موحّد** (`classify_result`): `مقبولة` فقط إن `consistent_sign=True`
  و`frac_significant` ≥ حدّ أدنى (0.34 افتراضياً — أي أكثر من ثلث النوافذ
  معنوية بنفس اتجاه المتوسط، دون تعسّف رقم أعلى بلا مبرّر). أقلّ من حدّ أدنى
  من النوافذ الناجحة (`n_ok`) → `قيد الاختبار` (لا حكم بعد). غير ذلك → `مرفوضة`.
* **تسجيل تلقائي**: كل (مرشّح × هدف) يُسجَّل في `experiment_registry` بمعرّف
  `SCAN_{اسم}_{هدف}` — **تمييز مهم**: هذه سجلّات آلية خفيفة من مسح دفعي، لا
  فرضيات مُنسَّقة يدوياً بعمق كـH001/H002 (تلك تبقى بمراجعة بشرية وتوثيق
  أوسع لكل شذوذ/ملاحظة). كلاهما في نفس السجلّ، والفائدة نفسها: يمنع اختبار
  نفس المرشّح مرتين لاحقاً بعد نسيان أنه فشل.

التسجيل اليدوي (بنمط H001/H002) يبقى الخيار الصحيح لأي فرضية تستحقّ تحليلاً
أعمق (شذوذ، مقارنة بخطّ أساس، تفسير سلوكي) — راجع تلك الدفاتر كمرجع للنمط.

In [ ]:
# @title
def classify_result(report, min_frac_significant=0.34, min_n_ok=5):
    """معيار قبول موحّد — نفس المنطق يُطبَّق بصرف النظر عن مصدر المرشّح
    (راجع "كيف تُقيَّم نتائج كل هذه الأدوات" في خطة المشروع). `n_ok` أقلّ من
    الحدّ الأدنى يعني عدد نوافذ ناجحة غير كافٍ للحكم أصلاً — لا "مرفوضة"
    مُتسرِّعة على دليل ضعيف."""
    if report.get("n_ok", 0) < min_n_ok:
        return "قيد الاختبار"
    if report.get("consistent_sign") and report.get("frac_significant", 0) >= min_frac_significant:
        return "مقبولة"
    return "مرفوضة"


def _batch_eval_job(job):
    cand, target, windows_, feature_order, eval_kwargs = job
    predict_fn = make_candidate_predict_fn(cand, feature_order=feature_order)
    try:
        report = evaluate_candidate(predict_fn, target, windows_, **eval_kwargs)
        return {"cand": cand, "target": target, "report": report, "status": "ok"}
    except Exception as e:
        return {"cand": cand, "target": target, "status": "error", "error": f"{type(e).__name__}: {e}"}


def run_batch_and_register(candidates, windows, targets=("close", "high", "low"), feature_order=None,
                           id_prefix="SCAN", max_workers=None, registry_path=None, **eval_kwargs):
    """المُشغّل الدفعي الكامل: يقيّم كل (مرشّح × هدف) بالتوازي عبر
    imap_ordered/default_workers (من دفتر التحضير)، يصنّف كل نتيجة عبر
    classify_result، ويسجّلها تلقائياً في experiment_registry. يُرجع
    (leaderboard, registered_ids) — الأولى للعرض السريع، والثانية لتتبّع ما
    كُتب فعلاً.

    ``registry_path``: مرّره (مثلاً tempfile) لتوجيه التسجيل بعيداً عن السجلّ
    الحقيقي — مفيد للاختبار الذاتي؛ اتركه ``None`` للمسار الافتراضي الحقيقي.
    """
    jobs = [(cand, target, windows, feature_order, eval_kwargs) for cand in candidates for target in targets]
    n_workers = max_workers or default_workers(len(jobs))

    rows, registered = [], []
    for res in imap_ordered(_batch_eval_job, jobs, max_workers=n_workers):
        cand, target = res["cand"], res["target"]
        if res["status"] == "error":
            rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                        "status": "error", "error": res["error"]})
            continue
        report = res["report"]
        status = classify_result(report)
        hyp_id = f"{id_prefix}_{cand['name']}_{target}"
        register_hypothesis(
            hyp_id=hyp_id,
            hypothesis=f"{cand.get('hypothesis', cand['name'])} (هدف: {target})",
            source=cand["track"],
            status=status,
            report={"per_window": report["per_window"], "mean_ic": report["mean_ic"],
                    "std_ic": report["std_ic"], "frac_significant": report["frac_significant"],
                    "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]},
            notes=(f"مُسجَّلة آلياً عبر run_batch_and_register. mean_ic={report['mean_ic']:.4f}, "
                  f"consistent_sign={report['consistent_sign']}, "
                  f"frac_significant={report['frac_significant']:.2f}."),
            registry_path=registry_path,
        )
        registered.append(hyp_id)
        rows.append({"name": cand["name"], "track": cand["track"], "target": target, "status": status,
                    "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                    "frac_significant": report["frac_significant"],
                    "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})

    df = pd.DataFrame(rows)
    ok = df[df["status"] != "error"].copy()
    if len(ok):
        ok["abs_mean_ic"] = ok["mean_ic"].abs()
        ok = ok.sort_values(["status", "abs_mean_ic"], ascending=[True, False])
        df = pd.concat([ok.drop(columns="abs_mean_ic"), df[df["status"] == "error"]], ignore_index=True)
    return df, registered


# مثال استخدام حقيقي (يكتب في experiment_registry الحقيقي — شغّله عمداً، لا تلقائياً):
# batch_leaderboard, registered_ids = run_batch_and_register(
#     CANDIDATE_SIGNALS + EXPLORATORY_CANDIDATES + GENERATIVE_CANDIDATES, windows, feature_order=FEATURE_ORDER)
# print(batch_leaderboard.to_string(index=False))
# print(f"سُجِّل {len(registered_ids)} مدخلاً: {registered_ids}")

### النتيجة الفعلية لتشغيل المُشغّل الدفعي الكامل (على السجلّ الحقيقي)

شُغِّل `run_batch_and_register` فعلياً (لا مثالاً معلَّقاً) على نفس بيانات
H002 (5 أصول، 12 نافذة) — **21 مرشّحاً قابلاً للتقييم × 3 أهداف = 63 مدخلاً
سُجِّلت في `experiment_registry/registry.json` الحقيقي** (المرشّح الثاني
والعشرون، `FUND_rate_extreme_position`، فشل بخطأ واضح على الأهداف الثلاثة
كما هو مصمَّم — `FUND_rate_z` غير مُفعَّلة في هذا الـdataset، فلم يُسجَّل).

**كل الـ64 مدخلاً (63 + `H002_nig_timenet_classification_head` سابقاً)
بحالة `مرفوضة`** — لا مرشّح واحد اجتاز `classify_result` (اتساق الاتجاه +
معنوية كافية) على هذه العيّنة من 5 أصول. هذا يُكمل **بوّابة خروج المرحلتين
١-٢** في خطة المشروع حرفياً ("توثيق كل نتيجة، مقبولة أو مرفوضة — لا حاجة
لنتيجة إيجابية للانتقال، فقط لدليل موثَّق"): بوّابة الخروج **اجتيزت
بالتوثيق**، لا بإيجاد إشارة. طبقاً لنص الخطة، هذا "مؤشّر مهم (لا نهائي) على
أن ميزات إضافية من نفس العائلة (OHLCV+مشتقّاتها المباشرة) لن تضيف كثيراً" —
يرفع أولوية إمّا (أ) توسيع عيّنة الأصول قبل الحكم النهائي (5 أصول عيّنة
صغيرة)، أو (ب) الانتقال للمرحلة ٣ في الخطة (Matrix Profile، SHAP
interactions، العنقدة — أدوات بلا قواعد مسبقة)، وليس تكرار مزيد من مرشّحين
من نفس العائلة. **قرار الانتقال يبقى صريحاً بيد صاحب المشروع، لا انزلاقاً
تلقائياً** (نفس مبدأ "الترتيب الزمني" في الخطة).

## ٩) اختبار ذاتي (بيانات تركيبية — بلا حاجة لـDrive)

يتحقّق من سلامة الوصلات (`clean_reg_target` يُزيل الأثر المصطنع فعلاً،
`scan_candidates` لا يتعطّل، الماسح يُرجع أعمدة اللوحة المتوقَّعة) على بيانات
عشوائية صغيرة — لا يثبت وجود إشارة حقيقية، فقط أن البنية تعمل.

In [ ]:
# @title
def run_discovery_lab_selftest():
    rng = np.random.default_rng(0)
    n_assets, n_per_asset, T, F = 3, 200, 8, len(FEATURE_ORDER) if 'FEATURE_ORDER' in globals() else 37
    feature_order = FEATURE_ORDER if 'FEATURE_ORDER' in globals() else [f"f{i}" for i in range(F)]
    n = n_assets * n_per_asset
    ts0 = pd.Timestamp("2022-01-01", tz="UTC")
    ts = pd.concat([pd.Series(pd.date_range(ts0, periods=n_per_asset, freq="1D"))
                    for _ in range(n_assets)], ignore_index=True)

    X = rng.normal(size=(n, T, F)).astype("float32")
    last_close = 100.0 * np.exp(rng.normal(scale=0.05, size=n).cumsum() / n_per_asset)
    body_idx = feature_order.index("BODY_ratio") if "BODY_ratio" in feature_order else 0
    # ✅ نزرع أثر مرجع "نفس النوع" عمداً: last_high يعتمد على BODY_ratio لا على
    #    حركة سعرية حقيقية — clean_reg_target يجب أن يُزيله، والهدف الخام (لو
    #    استُخدم بالخطأ) يجب أن يُظهره بوضوح.
    body_last = X[:, -1, body_idx]
    last_high = last_close * (1.0 + np.clip(-body_last, 0, None) * 0.05 + 1e-3)
    last_low = last_close * (1.0 - np.clip(body_last, 0, None) * 0.05 - 1e-3)
    future_high_max = last_close * (1.0 + rng.normal(scale=0.01, size=n))  # لا علاقة حقيقية بـbody_last
    future_low_min = last_close * (1.0 - np.abs(rng.normal(scale=0.01, size=n)))
    future_close = last_close * (1.0 + rng.normal(scale=0.01, size=n))

    y_high_reg_dirty = (future_high_max - last_high) / last_high  # مرجع "نفس النوع" (ملوَّث)
    y_low_reg_dirty = (future_low_min - last_low) / last_low
    y_close_reg = (future_close - last_close) / last_close

    last_candles = np.stack([last_high, last_low, last_close, ts.values.astype("int64"),
                             future_close, future_low_min, future_high_max], axis=1)
    flat = {"base_params": np.zeros((n, 2), "float32"), "last_candles": last_candles,
            "X_1D": X, "y": {"y_high_reg": y_high_reg_dirty, "y_low_reg": y_low_reg_dirty,
                             "y_close_reg": y_close_reg}}

    # ١) الحارس يُزيل الأثر المزروع فعلاً
    clean_high = clean_reg_target(flat, "high")
    from scipy.stats import spearmanr
    rho_dirty, _ = spearmanr(body_last, y_high_reg_dirty)
    rho_clean, _ = spearmanr(body_last, clean_high)
    assert abs(rho_dirty) > 0.3, f"الأثر المزروع ضعيف جداً للاختبار ({rho_dirty:.3f}) — أصلح البيانات التركيبية."
    assert abs(rho_clean) < abs(rho_dirty) / 3, (
        f"❌ clean_reg_target لم يُزل الأثر المزروع: dirty={rho_dirty:.3f} clean={rho_clean:.3f}")
    print(f"  ✅ clean_reg_target يُزيل أثر مرجع نفس النوع (dirty={rho_dirty:+.3f} → clean={rho_clean:+.3f})")

    # ٢) extract_feature_last_value / make_feature_predict_fn يعملان
    v = extract_feature_last_value(flat, feature_order[0], tf="1D", feature_order=feature_order)
    assert v.shape == (n,), "extract_feature_last_value: شكل خاطئ."
    predict_fn = make_feature_predict_fn(feature_order[0], transform=lambda x: -x,
                                         tf="1D", feature_order=feature_order)
    preds = predict_fn(flat, flat, flat)
    assert np.allclose(preds, -v), "make_feature_predict_fn: التحويل لم يُطبَّق بشكل صحيح."
    print("  ✅ extract_feature_last_value / make_feature_predict_fn تعملان بشكل صحيح")

    # ٢-ب) make_candidate_predict_fn: كل الأنواع الأربعة (feature/interaction/custom/trained)
    feat_b = feature_order[1] if len(feature_order) > 1 else feature_order[0]
    inter_cand = {"kind": "interaction", "feat_a": feature_order[0], "feat_b": feat_b, "op": "mul"}
    inter_fn = make_candidate_predict_fn(inter_cand, tf="1D", feature_order=feature_order)
    a = extract_feature_last_value(flat, feature_order[0], tf="1D", feature_order=feature_order)
    b = extract_feature_last_value(flat, feat_b, tf="1D", feature_order=feature_order)
    assert np.allclose(inter_fn(flat, flat, flat), a * b), "make_candidate_predict_fn: مسار التفاعل خاطئ."

    custom_cand = {"kind": "custom", "fn": lambda X_last, fo: X_last[:, 0] * 2.0}
    custom_fn = make_candidate_predict_fn(custom_cand, tf="1D", feature_order=feature_order)
    X_last_expected = extract_feature_matrix(flat, tf="1D", feature_order=feature_order)
    assert np.allclose(custom_fn(flat, flat, flat), X_last_expected[:, 0] * 2.0), (
        "make_candidate_predict_fn: مسار kind='custom' خاطئ.")

    trained_cand = {"kind": "trained", "builder": lambda feature_order: make_isolation_forest_predict_fn(
        [feature_order[0], feat_b], feature_order=feature_order, contamination=0.1)}
    trained_fn = make_candidate_predict_fn(trained_cand, feature_order=feature_order)
    trained_preds = trained_fn(flat, flat, flat)
    assert trained_preds.shape == (n,), "make_candidate_predict_fn: مسار kind='trained' أرجع شكلاً خاطئاً."
    print("  ✅ make_candidate_predict_fn يدعم الأنواع الأربعة (feature/interaction/custom/trained)")

    # ٣) scan_candidates يُرجع لوحة قيادة بالأعمدة المتوقَّعة، بلا انهيار
    windows_synth = [(flat, flat, flat)]
    tiny_candidates = [{"name": "f0", "track": "data_driven", "feature": feature_order[0], "transform": None}]
    board = scan_candidates(tiny_candidates, windows_synth, targets=("close", "high", "low"),
                            feature_order=feature_order, n_shuffles=20, min_samples=5)
    expected_cols = {"name", "track", "target", "status"}
    assert expected_cols.issubset(board.columns), f"أعمدة ناقصة في اللوحة: {board.columns.tolist()}"
    assert (board["status"] == "ok").all(), f"فشل تقييم بعض المرشّحين:\n{board}"
    print("  ✅ scan_candidates يُرجع لوحة قيادة سليمة بلا أخطاء")

    # ٤) classify_result: منطق حتمي بمعزل عن أي تدريب/عشوائية
    assert classify_result({"n_ok": 2, "consistent_sign": True, "frac_significant": 1.0}) == "قيد الاختبار", (
        "classify_result: n_ok قليل يجب أن يُرجع 'قيد الاختبار' بصرف النظر عن باقي الحقول")
    assert classify_result({"n_ok": 10, "consistent_sign": True, "frac_significant": 0.5}) == "مقبولة", (
        "classify_result: consistent_sign=True + frac_significant كافية يجب أن يُرجع 'مقبولة'")
    assert classify_result({"n_ok": 10, "consistent_sign": False, "frac_significant": 0.9}) == "مرفوضة", (
        "classify_result: consistent_sign=False يجب أن يُرجع 'مرفوضة' مهما كانت frac_significant")
    assert classify_result({"n_ok": 10, "consistent_sign": True, "frac_significant": 0.1}) == "مرفوضة", (
        "classify_result: frac_significant دون الحدّ الأدنى يجب أن يُرجع 'مرفوضة'")
    print("  ✅ classify_result يُطبِّق معيار القبول الموحّد بشكل صحيح (٤ حالات)")

    # ٥) run_batch_and_register: تسجيل فعلي إلى ملف مؤقّت (لا experiment_registry الحقيقي إطلاقاً)
    import tempfile, json as _json
    from pathlib import Path
    tmp_registry = Path(tempfile.mkdtemp()) / "registry_selftest.json"
    batch_board, registered_ids = run_batch_and_register(
        tiny_candidates, windows_synth, targets=("close", "high", "low"), feature_order=feature_order,
        id_prefix="SELFTEST", max_workers=2, registry_path=tmp_registry, n_shuffles=20, min_samples=5)
    assert len(registered_ids) == 3, f"يُتوقَّع تسجيل 3 (مرشّح واحد × 3 أهداف)، وُجد {len(registered_ids)}"
    assert tmp_registry.exists(), "run_batch_and_register: لم يُكتَب ملف السجلّ المؤقّت إطلاقاً"
    entries = _json.loads(tmp_registry.read_text(encoding="utf-8"))
    assert {e["id"] for e in entries} == set(registered_ids), "معرّفات السجلّ المكتوبة لا تطابق registered_ids"
    assert all(e["status"] in REGISTRY_STATUSES for e in entries), "حالة غير صالحة في سجلّ مكتوب فعلياً"
    assert all(e["id"].startswith("SELFTEST_") for e in entries), "id_prefix لم يُطبَّق على المعرّفات"
    print(f"  ✅ run_batch_and_register يقيّم بالتوازي (imap_ordered/default_workers) ويسجّل فعلياً "
          f"في ملف JSON مستقلّ ({len(registered_ids)} مدخلات، registry_path مُخصَّص لا الحقيقي)")

    print("✅ نجحت كل اختبارات مختبر بحث الإشارات الذاتية.")


run_discovery_lab_selftest()